In [3]:
import pandas as pd
import spacy
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter

# --- CONFIGURATION ---
RAW_DATA_CACHE = Path("./eval_cache/parsed_raw_data.csv")
nlp = spacy.load("en_core_web_sm")

# --- YOUR DATA LOADING FUNCTION ---
def parse_webnlg_multilingual(root_path):
    # ADD THIS LINE: Ensure the parent directory exists
    RAW_DATA_CACHE.parent.mkdir(parents=True, exist_ok=True)
    
    if RAW_DATA_CACHE.exists():
        print(f"Loading raw data from cache: {RAW_DATA_CACHE}")
        return pd.read_csv(RAW_DATA_CACHE, converters={'triples_en': eval})

    data = []
    root = Path(root_path)
    print("Parsing XML files...")
    for xml_file in tqdm(list(root.rglob("*.xml"))):
        tree = ET.parse(xml_file)
        for entry in tree.findall(".//entry"):
            eid, cat = entry.get("eid"), entry.get("category")
            triples_en = [t.text for t in entry.findall(".//modifiedtripleset/mtriple")]
            
            lex_map = {}
            for lex in entry.findall("lex"):
                lang, lid = lex.get("lang"), lex.get("lid")
                if lang not in lex_map: lex_map[lang] = {}
                lex_map[lang][lid] = lex.text
            
            if 'en' in lex_map:
                for target_lang in ['es', 'ca', "en_bt"]:
                    if target_lang in lex_map:
                        for lid, gold_text in lex_map['en'].items():
                            if lid in lex_map[target_lang]:
                                data.append({
                                    'eid': eid, 'category': cat, 'lid': lid,
                                    'lang': target_lang, 'text_en': gold_text,
                                    'text_target': lex_map[target_lang][lid],
                                    'triples_en': triples_en
                                })
    df = pd.DataFrame(data)
    df.to_csv(RAW_DATA_CACHE, index=False)
    return df

# --- LINGUISTIC FEATURE EXTRACTION ---
def get_mtese_features(text):
    doc = nlp(str(text))
    tokens = [t.text.lower() for t in doc if not t.is_punct]
    
    # 1. Vocabulary Diversity (TTR)
    ttr = len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0
    
    # 2. Lexical Density (Content vs Function words)
    content_pos = {'NOUN', 'VERB', 'ADJ', 'ADV'}
    content_words = sum(1 for t in doc if t.pos_ in content_pos)
    density = content_words / len(tokens) if len(tokens) > 0 else 0
    
    # 3. Syntactic Complexity (Mean Dependency Distance)
    # Measures how "linear" vs "nested" a sentence is
    dep_dist = np.mean([abs(t.i - t.head.i) for t in doc])
    
    # 4. Sentence Length (Tokens)
    length = len(tokens)
    
    return {
        'diversity_ttr': ttr,
        'lexical_density': density,
        'syntactic_complexity': dep_dist,
        'token_count': length
    }

# --- COMPARISON LOGIC ---
def analyze_en_vs_enbt_gap(df):
    # Filter only the rows containing back-translations
    bt_subset = df[df['lang'] == 'en_bt'].copy()
    
    print(f"Analyzing {len(bt_subset)} instance pairs for Machine Translationese...")
    
    # Extract features for both columns
    gold_feats = pd.DataFrame([get_mtese_features(t) for t in tqdm(bt_subset['text_en'], desc="Processing Gold")])
    bt_feats = pd.DataFrame([get_mtese_features(t) for t in tqdm(bt_subset['text_target'], desc="Processing BT")])
    
    results = []
    for col in gold_feats.columns:
        g_mean = gold_feats[col].mean()
        b_mean = bt_feats[col].mean()
        delta = ((b_mean - g_mean) / g_mean) * 100
        
        results.append({
            'Metric': col.replace('_', ' ').title(),
            'Gold English': round(g_mean, 4),
            'Back-Translated': round(b_mean, 4),
            'Gap (%)': round(delta, 2)
        })
        
    return pd.DataFrame(results)

In [4]:
df_raw = parse_webnlg_multilingual("../WebNLG_CA_BT")
mtese_results = analyze_en_vs_enbt_gap(df_raw)
    
print("\n--- LINGUISTIC GAP ANALYSIS (GOLD vs BT) ---")
print(mtese_results.to_string(index=False))

Loading raw data from cache: eval_cache/parsed_raw_data.csv
Analyzing 51240 instance pairs for Machine Translationese...


Processing Gold:   0%|          | 0/51240 [00:00<?, ?it/s]

Processing BT:   0%|          | 0/51240 [00:00<?, ?it/s]


--- LINGUISTIC GAP ANALYSIS (GOLD vs BT) ---
              Metric  Gold English  Back-Translated  Gap (%)
       Diversity Ttr        0.8890           0.8880    -0.12
     Lexical Density        0.2543           0.2542    -0.06
Syntactic Complexity        2.3957           2.4255     1.24
         Token Count       20.4533          20.1227    -1.62
